# 0ScrapingBerita

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time
import os

def generate_date_range(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    return [(start + timedelta(days=i)) for i in range((end - start).days + 1)]

def get_next_id(filename):
    """Dapatkan ID terakhir dari CSV yang sudah ada"""
    if os.path.exists(filename):
        try:
            df = pd.read_csv(filename)
            if len(df) > 0:
                return df['id'].max() + 1
        except:
            pass
    return 1

def save_to_csv(data, filename, mode='a'):
    """Simpan data ke CSV secara incremental"""
    df = pd.DataFrame([data])

    # Jika file belum ada, buat dengan header
    if not os.path.exists(filename):
        df.to_csv(filename, index=False, quoting=1, mode='w')
    else:
        # Append tanpa header
        df.to_csv(filename, index=False, quoting=1, mode='a', header=False)

def scrape_detik_by_date(date_obj):
    """Scrape berita Detik Finance untuk 1 tanggal"""
    date_mdy = date_obj.strftime("%m/%d/%Y")
    date_ymd = date_obj.strftime("%Y-%m-%d")
    print(f"📅 Scraping tanggal {date_ymd}")

    all_articles = []
    headers = {'User-Agent': 'Mozilla/5.0'}

    for page in range(1, 11):  # maksimal 10 halaman
        url = f"https://finance.detik.com/indeks?page={page}&date={date_mdy}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        articles = soup.find_all("article", class_="list-content__item")
        if not articles:
            if page == 1:
                print("   ⚠️ Tidak ada berita ditemukan di tanggal ini.")
            else:
                print(f"   Berhenti di halaman ke-{page}, tidak ada berita lagi.")
            break

        print(f"   Halaman {page} → {len(articles)} berita")

        for art in articles:
            try:
                title_tag = art.find("h3", class_="media__title").find("a")
                title = title_tag.text.strip()
                link = title_tag["href"]
                all_articles.append({
                    "tanggal": date_ymd,
                    "judul": title,
                    "link": link
                })
            except:
                continue
        time.sleep(1)

    return all_articles

def scrape_detik_content(link):
    """Ambil isi artikel dari link berita"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
    }
    try:
        res = requests.get(link, headers=headers, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')

        # Cari container utama artikel
        body = soup.select_one("div.detail__body-text.itp_bodycontent")

        if not body:
            body = soup.select_one("div.detail__body-text")

        if not body:
            print(f"      ⚠️ Container artikel tidak ditemukan")
            return ""

        # Hapus elemen yang PASTI bukan konten artikel
        for unwanted in body.select('.parallaxindetail, .linksisip, .aevp, script, style, .sisip_video_ds, .detail__body-tag, .skybanner, iframe, .ads-scrollpage-container, .pip-vid'):
            unwanted.decompose()

        # Ambil semua teks, lalu bersihkan
        full_text = body.get_text(separator=' ', strip=True)

        # Hapus semua noise text
        noise_patterns = [
            "SCROLL TO CONTINUE WITH CONTENT",
            "ADVERTISEMENT",
            "Advertisement",
            "Baca juga:",
            "Simak Video",
            "Video:",
            "Simak juga",
            "Lihat juga:",
            "(/4905536/",
            "google_ads_iframe"
        ]

        for pattern in noise_patterns:
            full_text = full_text.replace(pattern, "")

        # Bersihkan whitespace berlebih
        import re
        full_text = re.sub(r'\s+', ' ', full_text).strip()

        if not full_text or len(full_text) < 50:
            print(f"      ⚠️ Konten terlalu pendek: {len(full_text)} karakter")
            return ""

        return full_text

    except Exception as e:
        print(f"      ⚠️ Error mengambil konten: {e}")
        return ""

def scrape_detik_range(start_date, end_date, output_file="detik_scraping.csv"):
    print(f"Mulai scraping dari {start_date} sampai {end_date}")
    print(f"File output: {output_file}\n")

    # Dapatkan ID awal (lanjutkan dari data yang sudah ada)
    id_counter = get_next_id(output_file)

    if os.path.exists(output_file):
        print(f"File sudah ada. Melanjutkan dari ID {id_counter}\n")

    total_saved = 0

    for date_obj in generate_date_range(start_date, end_date):
        try:
            articles = scrape_detik_by_date(date_obj)
            print(f"✅ {len(articles)} berita ditemukan.\n")

            for art in articles:
                try:
                    print(f"[{id_counter}] Mengambil: {art['judul'][:60]}...")
                    isi = scrape_detik_content(art['link'])

                    # Simpan langsung ke CSV
                    data = {
                        "id": id_counter,
                        "tanggal": art['tanggal'],
                        "judul": art['judul'],
                        "isi": isi
                    }
                    save_to_csv(data, output_file)

                    id_counter += 1
                    total_saved += 1

                    time.sleep(2)  # Delay lebih lama agar konten sempat load

                except Exception as e:
                    print(f"       Error pada artikel: {e}")
                    continue

        except Exception as e:
            print(f" Error pada tanggal {date_obj.strftime('%Y-%m-%d')}: {e}")
            continue

    print(f"\n Selesai! Total {total_saved} berita tersimpan di {output_file}")

# ==== Jalankan ====
if __name__ == "__main__":
    scrape_detik_range(
        start_date="2022-05-02",
        end_date="2022-07-01", #year-month-date
        output_file="detikv2_020522-010722.csv"
    )